### Importamos las librerias necesarias y importamos la demanda

In [18]:
!pip install pulp

In [19]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import warnings
import math

In [25]:
# -------------------------------------------------------------
# FASE 1: LIMPIEZA DE DATOS (Todos los SKUs)
# -------------------------------------------------------------
df = pd.read_csv('historial_ventas_lavandina.csv')

df['Fecha'] = pd.to_datetime(df['Fecha'])
df['Mes_Anio'] = df['Fecha'].dt.to_period('M')

# Agrupamos sumando los litros por mes y por SKU
df_agrupado = df.groupby(['Mes_Anio', 'SKU'])['Litros_Vendidos'].sum().reset_index()



### Analizamos el pronostico, de cada sku

In [26]:
# -------------------------------------------------------------
# FASE 2: MOTOR DE PRONÓSTICO BATCH
# -------------------------------------------------------------
# Vamos a guardar la demanda del mes que queremos planificar (Noviembre 2026), formando un listado con sky y su respectiva demanda
# -------------------------------------------------------------

mes_objetivo = '2026-11-01'
demanda_noviembre = {}

print(f"--- PRONÓSTICO DE DEMANDA: {mes_objetivo} ---")

# El bucle 'for' recorre los 3 SKUs de forma automática

for sku in df_agrupado['SKU'].unique():
    
    # 1. Aislamos los datos del SKU actual
    
    serie_sku = df_agrupado[df_agrupado['SKU'] == sku].copy()
    serie_sku['Mes_Anio'] = serie_sku['Mes_Anio'].dt.to_timestamp()
    serie_sku.set_index('Mes_Anio', inplace=True)
    serie_historica = serie_sku['Litros_Vendidos']
    
    # 2. Entrenamos el modelo matemático
    modelo = ExponentialSmoothing(serie_historica, trend='add', seasonal='add', seasonal_periods=12).fit()
    
    # 3. Proyectamos y extraemos específicamente el volumen de noviembre
    pronostico_futuro = modelo.forecast(6)
    volumen_proyectado = pronostico_futuro.loc[mes_objetivo]
    
    # 4. Guardamos el resultado en nuestro diccionario
    demanda_noviembre[sku] = round(volumen_proyectado)
    
    print(f"SKU: {sku} -> {demanda_noviembre[sku]:,.0f} Litros")

--- PRONÓSTICO DE DEMANDA: 2026-11-01 ---
SKU: LAV-BLA-03 -> 85,454 Litros
SKU: LAV-CLA-01 -> 327,721 Litros
SKU: LAV-FLO-02 -> 159,074 Litros


/home/eliseo/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/home/eliseo/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/holtwinters/model.py:917: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(
/home/eliseo/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/home/eliseo/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/holtwinters/model.py:917: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(
/home/eliseo/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(da

### Realizamos el MRP, conociendo las recetas y tiempos para la producción de cada articulo

In [27]:
# 1. Demanda dinámica (Viene de la Fase 2)
demanda_dinamica = demanda_noviembre

asignacion_tanques = {'LAV-CLA-01': 'TK-100', 'LAV-FLO-02': 'TK-200', 'LAV-BLA-03': 'TK-200'}
costo_hora_operario = 5000.0  
costo_kwh = 120.0             
horas_mezclado = 3.0          

# 2. Leemos AMBOS archivos maestros
df_equipos = pd.read_csv('maestro_equipos.csv')
df_bom = pd.read_csv('maestro_recetas.csv')  

fecha = pd.to_datetime(mes_objetivo)

print(f"--- ANÁLISIS FINANCIERO TOTAL (OPEX + MATERIA PRIMA) --- {fecha.strftime('%m/%Y')}")

costo_global_planta = 0

# ---> SE REALIZA UN LOOP PARA ARMAR UNA LISTA CON LOS LOTES A REALIZAR
lotes_a_programar = []

for sku, demanda in demanda_dinamica.items():
    # --- DATOS DEL EQUIPO ---
    tanque = asignacion_tanques[sku]
    info_tanque = df_equipos[df_equipos['ID_Tanque'] == tanque].iloc[0]
    capacidad = info_tanque['Capacidad_L']
    potencia_motor = info_tanque['Potencia_Agitador_kW']
    tiempo_lavado = info_tanque['Tiempo_CIP_hs']
    costo_quimicos_cip = info_tanque['Costo_CIP_ARS']
    
    lotes = math.ceil(demanda / capacidad)
    
    # --- 1. CÁLCULO DE OPEX (Fijo por Lote) ---
    horas_totales_batch = horas_mezclado + tiempo_lavado
    costo_labor_batch = horas_totales_batch * costo_hora_operario
    costo_energia_batch = horas_mezclado * potencia_motor * costo_kwh
    opex_batch = costo_labor_batch + costo_energia_batch + costo_quimicos_cip

    # ---> NUEVO: GUARDAMOS LOS LOTES FÍSICOS PARA LA FASE 4
    for i in range(lotes):
        lotes_a_programar.append({
            'Lote': f"{sku}_B{i+1}",
            'Tanque': tanque,
            'Duracion_hs': horas_totales_batch
        })
    
    # --- 2. CÁLCULO DE MATERIA PRIMA (Variable según receta) ---
    receta = df_bom[df_bom['SKU_Producto'] == sku]
    costo_mp_batch = 0
    
    # Iteramos sobre los ingredientes que lleva un tanque entero
    for idx, fila in receta.iterrows():
        litros_insumo = capacidad * fila['Fraccion_Receta']
        costo_insumo = litros_insumo * fila['Costo_Unitario_ARS']
        costo_mp_batch += costo_insumo

    # --- 3. COSTO TOTAL DEL BATCH ---
    costo_total_batch = opex_batch + costo_mp_batch
    inversion_total_sku = costo_total_batch * lotes
    costo_global_planta += inversion_total_sku
    
    print(f"\nSKU: {sku} | {lotes} Batch(es) en {tanque} ({capacidad} L c/u)")
    print(f"  * Costo Materia Prima (Reactivos): ${costo_mp_batch:,.2f} por lote")
    print(f"  * Costo OPEX (Energía, Labor, CIP): ${opex_batch:,.2f} por lote")
    print(f"  -> Costo Total de un Lote: ${costo_total_batch:,.2f}")
    print(f"  -> INVERSIÓN MENSUAL PARA CUBRIR DEMANDA: ${inversion_total_sku:,.2f}")

print("\n" + "="*60)
print(f"PRESUPUESTO TOTAL REQUERIDO PARA EL MES: ${costo_global_planta:,.2f}")
print(f"\n[Sistema] Se generaron {len(lotes_a_programar)} lotes en memoria listos.")
print(lotes_a_programar)

--- ANÁLISIS FINANCIERO TOTAL (OPEX + MATERIA PRIMA) --- 11/2026

SKU: LAV-BLA-03 | 18 Batch(es) en TK-200 (5000 L c/u)
  * Costo Materia Prima (Reactivos): $636,375.00 por lote
  * Costo OPEX (Energía, Labor, CIP): $37,380.00 por lote
  -> Costo Total de un Lote: $673,755.00
  -> INVERSIÓN MENSUAL PARA CUBRIR DEMANDA: $12,127,590.00

SKU: LAV-CLA-01 | 33 Batch(es) en TK-100 (10000 L c/u)
  * Costo Materia Prima (Reactivos): $717,500.00 por lote
  * Costo OPEX (Energía, Labor, CIP): $48,400.00 por lote
  -> Costo Total de un Lote: $765,900.00
  -> INVERSIÓN MENSUAL PARA CUBRIR DEMANDA: $25,274,700.00

SKU: LAV-FLO-02 | 32 Batch(es) en TK-200 (5000 L c/u)
  * Costo Materia Prima (Reactivos): $716,750.00 por lote
  * Costo OPEX (Energía, Labor, CIP): $37,380.00 por lote
  -> Costo Total de un Lote: $754,130.00
  -> INVERSIÓN MENSUAL PARA CUBRIR DEMANDA: $24,132,160.00

PRESUPUESTO TOTAL REQUERIDO PARA EL MES: $61,534,450.00

[Sistema] Se generaron 83 lotes en memoria listos.
[{'Lote': 'L

### Optimización y Secuenciación de Planta con Pyomo

In [28]:
import pulp
import itertools

# =============================================================
# 1. PREPARACIÓN DE LOS DATOS (Python puro)
# =============================================================
# PuLP necesita listas y diccionarios simples para construir las ecuaciones.

lotes_tk100 = [item['Lote'] for item in lotes_a_programar if item['Tanque'] == 'TK-100']
lotes_tk200 = [item['Lote'] for item in lotes_a_programar if item['Tanque'] == 'TK-200']

# duracion es un diccionario que usaremos como constante en nuestras ecuaciones.
# Ejemplo: {'LAV-CLA-01': 4.5}
duracion = {item['Lote']: item['Duracion_hs'] for item in lotes_a_programar}
todos_los_lotes = list(duracion.keys())

# =============================================================
# 2. INICIALIZACIÓN DEL MODELO MATEMÁTICO
# =============================================================
# pulp.LpProblem crea el "contenedor" vacío donde vamos a meter nuestras reglas.
# El segundo parámetro define si queremos MAXIMIZAR (ej. ganancias) o MINIMIZAR (ej. tiempos o costos).
# Usamos pulp.LpMinimize porque queremos que el tiempo total de planta sea el menor posible.
modelo = pulp.LpProblem("Secuenciacion_Reactores", pulp.LpMinimize)


# =============================================================
# 3. CREACIÓN DE VARIABLES DE DECISIÓN (Las incógnitas)
# =============================================================
# Las variables son los valores que PuLP tiene permiso para modificar y calcular.

# pulp.LpVariable.dicts crea múltiples variables a la vez usando una lista de nombres (todos_los_lotes).
# lowBound=0 es vital: le dice a PuLP que la hora de inicio no puede ser negativa (no existe la hora -5).
# cat='Continuous' indica que el resultado puede tener decimales (ej. arranca en la hora 3.5).
arranque = pulp.LpVariable.dicts("Inicio", todos_los_lotes, lowBound=0, cat='Continuous')

# pulp.LpVariable crea una sola variable suelta. Representa la hora final de toda la campaña.
makespan = pulp.LpVariable("Makespan_Total", lowBound=0, cat='Continuous')

# Constante Big-M para las restricciones lógicas.
M = sum(duracion.values()) 


# =============================================================
# 4. CONSTRUCCIÓN DE RESTRICCIONES (El operador +=)
# =============================================================
# En PuLP, las restricciones se agregan inyectándolas al "modelo" usando el operador "+=".
# Cada vez que hacés "modelo += [ecuación]", estás sumando una regla física obligatoria.

if len(lotes_tk100) > 1:
    pares_100 = list(itertools.combinations(lotes_tk100, 2))
    
    # Creamos variables Binarias (cat='Binary'). PuLP solo podrá asignarles el valor 0 o el valor 1.
    # Estas variables actúan como llaves lógicas para decidir qué lote entra primero al equipo.
    y100 = pulp.LpVariable.dicts("Orden100", pares_100, cat='Binary')
    
    for i, j in pares_100:
        if i[:-3] == j[:-3]: 
            # RESTRICCIÓN RÍGIDA: El inicio de J tiene que ser MAYOR O IGUAL al inicio de I + lo que dura I.
            # Como usamos el operador +=, PuLP entiende que esto es una regla que no puede romper jamás.
            modelo += arranque[j] >= arranque[i] + duracion[i]
        else:
            # RESTRICCIONES MUTUAMENTE EXCLUYENTES (Método Big-M)
            # Solo una de estas dos ecuaciones será matemáticamente relevante, dependiendo de si y100 vale 0 o 1.
            # La otra ecuación quedará con un número negativo gigante en la derecha (gracias a M) y PuLP la ignorará.
            modelo += arranque[j] >= arranque[i] + duracion[i] - M * (1 - y100[(i,j)])
            modelo += arranque[i] >= arranque[j] + duracion[j] - M * y100[(i,j)]

if len(lotes_tk200) > 1:
    pares_200 = list(itertools.combinations(lotes_tk200, 2))
    y200 = pulp.LpVariable.dicts("Orden200", pares_200, cat='Binary')
    
    for i, j in pares_200:
        if i[:-3] == j[:-3]:
            modelo += arranque[j] >= arranque[i] + duracion[i]
        else:
            modelo += arranque[j] >= arranque[i] + duracion[i] - M * (1 - y200[(i,j)])
            modelo += arranque[i] >= arranque[j] + duracion[j] - M * y200[(i,j)]

# RESTRICCIÓN DE MAKESPAN
for lote in todos_los_lotes:
    # Le decimos a PuLP que la variable "makespan" tiene que ser mayor o igual 
    # a la hora en que termina cada uno de los lotes. 
    modelo += makespan >= arranque[lote] + duracion[lote]


# =============================================================
# 5. DEFINICIÓN DE LA FUNCIÓN OBJETIVO
# =============================================================
# Cuando sumás una variable (o ecuación) al modelo SIN el signo de igualdad o desigualdad (>=, <=, ==),
# PuLP automáticamente asume que esa es la Función Objetivo.
# Como al principio declaramos LpMinimize, PuLP intentará aplastar el valor de esta variable al mínimo posible.
modelo += makespan


# =============================================================
# 6. EJECUCIÓN DEL SOLVER
# =============================================================
# PuLP es solo un traductor. El que realmente hace la matemática es el "Solver" (por defecto, CBC de COIN-OR).
# pulp.PULP_CBC_CMD() nos permite pasarle parámetros al motor, como el tiempo máximo.
solver = pulp.PULP_CBC_CMD(timeLimit=60, msg=False)

# El método solve() congela el modelo, se lo envía al solver, y espera la respuesta.
modelo.solve(solver)


# =============================================================
# 7. EXTRACCIÓN DE LOS RESULTADOS
# =============================================================
cronograma = []

for lote in todos_los_lotes:
    tanque = 'TK-100' if lote in lotes_tk100 else 'TK-200'
    
    # pulp.value() es la llave maestra final. 
    # Toma una variable matemática abstracta y te devuelve el número concreto (float) que el solver decidió.
    hora_inicio = pulp.value(arranque[lote])
    hora_fin = hora_inicio + duracion[lote]
    
    cronograma.append({
        'Tanque': tanque, 
        'Lote': lote, 
        'Inicio (hs)': hora_inicio, 
        'Fin (hs)': hora_fin
    })

df_gantt = pd.DataFrame(cronograma).sort_values(by=['Tanque', 'Inicio (hs)'])
df_gantt.to_csv('cronograma_produccion_pulp.csv', index=False)